# Generative Adversarial Nets — Implementation / 구현

**Paper**: I. J. Goodfellow et al., "Generative Adversarial Nets," *NIPS*, 2014.

This notebook implements the core GAN algorithm from the paper.
이 노트북은 논문의 핵심 GAN 알고리즘을 구현합니다.

1. **Part 1**: 1D Gaussian GAN — 이론적 직관을 위한 단순한 1D 예제 / Simple 1D example for theoretical intuition (Figure 1 재현)
2. **Part 2**: MNIST GAN — 논문의 실험을 재현하는 MLP 기반 GAN / MLP-based GAN reproducing the paper's experiments
3. **Part 3**: 이론 검증 — 최적 Discriminator와 JSD 수렴 / Theory verification — optimal D and JSD convergence

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: 1D Gaussian GAN / 1D 가우시안 GAN

이 파트는 논문의 Figure 1을 재현합니다. 단순한 1D 가우시안 분포를 학습하는 GAN을 구현하여
G, D, $p_g$의 학습 과정을 시각화합니다.

This part reproduces Figure 1 from the paper. We implement a GAN that learns a simple 1D Gaussian
distribution, visualizing the training dynamics of G, D, and $p_g$.

**목표 / Goal**: $p_{\text{data}} = \mathcal{N}(\mu=4, \sigma=1.25)$를 학습

In [ ]:
# --- 1D GAN: Generator and Discriminator ---

class Generator1D(nn.Module):
    """Generator for 1D data.

    Maps noise z ~ U(0,1) to data space x via a small MLP.
    """

    def __init__(self, input_dim: int = 1, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class Discriminator1D(nn.Module):
    """Discriminator for 1D data.

    Outputs probability that input x is from real data distribution.
    """

    def __init__(self, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# Data distribution: N(mu=4, sigma=1.25)
DATA_MU = 4.0
DATA_SIGMA = 1.25

def sample_real_data(n: int) -> torch.Tensor:
    """Sample from the real data distribution p_data = N(4, 1.25^2)."""
    return torch.randn(n, 1) * DATA_SIGMA + DATA_MU

def sample_noise(n: int, dim: int = 1) -> torch.Tensor:
    """Sample noise z ~ U(0, 1)."""
    return torch.rand(n, dim)

In [ ]:
# --- 1D GAN Training (Algorithm 1 from the paper) ---

def train_1d_gan(
    n_epochs: int = 5000,
    batch_size: int = 256,
    k: int = 1,
    lr: float = 1e-3,
    snapshot_epochs: list | None = None,
) -> dict:
    """Train a 1D GAN following Algorithm 1.

    Args:
        n_epochs: Number of training iterations.
        batch_size: Minibatch size m.
        k: Number of D update steps per G update step.
        lr: Learning rate.
        snapshot_epochs: Epochs at which to save snapshots for visualization.

    Returns:
        Dictionary with training history and snapshots.
    """
    if snapshot_epochs is None:
        snapshot_epochs = [0, 100, 500, 5000]

    G = Generator1D().to(device)
    D = Discriminator1D().to(device)

    # Paper used momentum; Adam is the modern standard equivalent
    opt_g = optim.Adam(G.parameters(), lr=lr)
    opt_d = optim.Adam(D.parameters(), lr=lr)

    criterion = nn.BCELoss()
    history = {"d_loss": [], "g_loss": [], "snapshots": {}}

    # Snapshot at epoch 0 (before any training)
    if 0 in snapshot_epochs:
        with torch.no_grad():
            x_range = torch.linspace(-2, 10, 1000).unsqueeze(1).to(device)
            d_values = D(x_range).cpu().numpy().flatten()
            gen_samples = G(sample_noise(10000).to(device)).cpu().numpy().flatten()
        history["snapshots"][0] = {
            "x_range": x_range.cpu().numpy().flatten(),
            "d_values": d_values,
            "gen_samples": gen_samples,
        }

    for epoch in range(1, n_epochs + 1):
        # --- Discriminator training (k steps) ---
        for _ in range(k):
            # Sample minibatch of m noise samples from p_z
            z = sample_noise(batch_size).to(device)
            # Sample minibatch of m examples from p_data
            real = sample_real_data(batch_size).to(device)
            fake = G(z).detach()

            # D wants to maximize: log D(x) + log(1 - D(G(z)))
            d_real = D(real)
            d_fake = D(fake)
            d_loss = criterion(d_real, torch.ones_like(d_real)) + \
                     criterion(d_fake, torch.zeros_like(d_fake))

            opt_d.zero_grad()
            d_loss.backward()
            opt_d.step()

        # --- Generator training (1 step) ---
        z = sample_noise(batch_size).to(device)
        fake = G(z)
        d_fake = D(fake)

        # Practical trick: maximize log D(G(z)) instead of
        # minimizing log(1 - D(G(z))) to avoid gradient saturation
        g_loss = criterion(d_fake, torch.ones_like(d_fake))

        opt_g.zero_grad()
        g_loss.backward()
        opt_g.step()

        history["d_loss"].append(d_loss.item())
        history["g_loss"].append(g_loss.item())

        # Save snapshots for Figure 1 visualization
        if epoch in snapshot_epochs:
            with torch.no_grad():
                x_range = torch.linspace(-2, 10, 1000).unsqueeze(1).to(device)
                d_values = D(x_range).cpu().numpy().flatten()
                gen_samples = G(sample_noise(10000).to(device)).cpu().numpy().flatten()
            history["snapshots"][epoch] = {
                "x_range": x_range.cpu().numpy().flatten(),
                "d_values": d_values,
                "gen_samples": gen_samples,
            }

    return history


history_1d = train_1d_gan(n_epochs=5000, k=1, snapshot_epochs=[0, 100, 500, 5000])
print("Training complete.")

In [ ]:
### Figure 1 재현: 학습 과정 시각화 / Reproducing Figure 1: Training Dynamics

논문의 Figure 1처럼 학습 과정의 네 단계를 시각화합니다:
- **검은 점선**: $p_{\text{data}}$ (실제 데이터 분포 / real data distribution)
- **초록 히스토그램**: $p_g$ (생성된 데이터 분포 / generated data distribution)
- **파란 파선**: $D(x)$ (Discriminator 출력 / Discriminator output)

학습이 진행되면서 초록색 $p_g$가 검은색 $p_{\text{data}}$에 수렴하고,
파란색 $D(x)$가 0.5로 평탄해지는 것을 확인합니다.

As training progresses, the green $p_g$ converges to the black $p_{\text{data}}$,
and the blue $D(x)$ flattens to 0.5.

In [ ]:
# --- Figure 1 reproduction: Training dynamics ---
from scipy.stats import norm

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
snapshot_epochs = [0, 100, 500, 5000]
titles = ["(a) Before training", "(b) Early training",
          "(c) Mid training", "(d) Converged"]

x_plot = np.linspace(-2, 10, 1000)
p_data = norm.pdf(x_plot, loc=DATA_MU, scale=DATA_SIGMA)

for ax, epoch, title in zip(axes, snapshot_epochs, titles):
    snap = history_1d["snapshots"][epoch]

    # p_data (black dotted)
    ax.plot(x_plot, p_data, 'k:', linewidth=2, label=r'$p_{\mathrm{data}}$')

    # p_g (green solid) — histogram of generated samples
    ax.hist(snap["gen_samples"], bins=80, range=(-2, 10), density=True,
            color='green', alpha=0.5, label=r'$p_g$ (G)')

    # D(x) (blue dashed) — on secondary y-axis
    ax2 = ax.twinx()
    ax2.plot(snap["x_range"], snap["d_values"], 'b--', linewidth=1.5,
             label='$D(x)$')
    ax2.set_ylim(-0.1, 1.1)
    ax2.axhline(y=0.5, color='gray', linestyle=':', alpha=0.3)
    if epoch == snapshot_epochs[-1]:
        ax2.set_ylabel('$D(x)$', color='blue')

    ax.set_title(title)
    ax.set_xlim(-2, 10)
    ax.set_ylim(0, 0.6)
    if epoch == snapshot_epochs[0]:
        ax.set_ylabel('Density')
    ax.set_xlabel('$x$')

# Add legend
axes[0].legend(loc='upper left', fontsize=9)
fig.suptitle("Figure 1 Reproduction: GAN Training Dynamics on 1D Gaussian",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Training loss curves ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_1d["d_loss"], label="D loss", alpha=0.7)
ax.plot(history_1d["g_loss"], label="G loss", alpha=0.7)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("1D GAN Training Loss")
ax.legend()

# At equilibrium, both losses should stabilize around -log(4) ≈ 1.386
ax.axhline(y=np.log(4), color='gray', linestyle=':', alpha=0.5,
           label=f'-log(4) ≈ {np.log(4):.3f}')
plt.tight_layout()
plt.show()

## Part 2: MNIST GAN / MNIST GAN

논문 Section 5의 실험을 재현합니다. MLP 기반 Generator와 Discriminator로 MNIST 손글씨 숫자를 생성합니다.
논문에서는 Generator에 ReLU + sigmoid, Discriminator에 maxout + dropout을 사용했습니다.
여기서는 현대적 등가물인 LeakyReLU와 dropout을 사용합니다.

Reproducing the experiments from Section 5. MLP-based Generator and Discriminator generate
MNIST handwritten digits. The paper used ReLU + sigmoid for G and maxout + dropout for D.
We use the modern equivalents: LeakyReLU and dropout.

- **Generator**: $z \in \mathbb{R}^{100} \sim \mathcal{N}(0, 1)$ → MLP → $x \in \mathbb{R}^{784}$ (28×28 image)
- **Discriminator**: $x \in \mathbb{R}^{784}$ → MLP + dropout → probability $\in [0, 1]$

In [ ]:
# --- MNIST GAN: Model definitions ---

LATENT_DIM = 100   # Dimension of noise vector z
IMG_DIM = 784      # 28 * 28 flattened MNIST image


class GeneratorMNIST(nn.Module):
    """MLP Generator for MNIST (28x28 images).

    Maps latent noise z (dim=100) to a 784-dimensional image vector.
    Uses ReLU + sigmoid output as in the original paper.
    """

    def __init__(self, latent_dim: int = LATENT_DIM, hidden_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim * 2, IMG_DIM),
            nn.Sigmoid(),  # Output in [0, 1] to match normalized images
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class DiscriminatorMNIST(nn.Module):
    """MLP Discriminator for MNIST.

    Paper uses maxout activations + dropout.
    We use LeakyReLU + dropout as the modern equivalent.
    """

    def __init__(self, hidden_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(IMG_DIM, hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),     # Dropout as in the paper
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
# --- MNIST GAN Training ---

def train_mnist_gan(
    n_epochs: int = 50,
    batch_size: int = 128,
    k: int = 1,
    lr: float = 2e-4,
) -> tuple:
    """Train GAN on MNIST following Algorithm 1.

    Args:
        n_epochs: Number of training epochs.
        batch_size: Minibatch size.
        k: D update steps per G update step.
        lr: Learning rate.

    Returns:
        Tuple of (G, D, history dict).
    """
    # Load MNIST dataset
    transform = transforms.Compose([
        transforms.ToTensor(),  # Scales to [0, 1]
    ])
    dataset = datasets.MNIST(root='./data', train=True, download=True,
                             transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                            drop_last=True)

    G = GeneratorMNIST().to(device)
    D = DiscriminatorMNIST().to(device)

    opt_g = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

    criterion = nn.BCELoss()
    history = {"d_loss": [], "g_loss": [], "samples": []}

    for epoch in range(1, n_epochs + 1):
        epoch_d_loss = 0.0
        epoch_g_loss = 0.0
        n_batches = 0

        for real_imgs, _ in dataloader:
            real_imgs = real_imgs.view(-1, IMG_DIM).to(device)
            batch_sz = real_imgs.size(0)

            # --- Train Discriminator (k steps) ---
            for _ in range(k):
                z = torch.randn(batch_sz, LATENT_DIM).to(device)
                fake_imgs = G(z).detach()

                d_real = D(real_imgs)
                d_fake = D(fake_imgs)

                d_loss = criterion(d_real, torch.ones_like(d_real)) + \
                         criterion(d_fake, torch.zeros_like(d_fake))

                opt_d.zero_grad()
                d_loss.backward()
                opt_d.step()

            # --- Train Generator (1 step) ---
            z = torch.randn(batch_sz, LATENT_DIM).to(device)
            fake_imgs = G(z)
            d_fake = D(fake_imgs)

            # Practical trick: max log D(G(z))
            g_loss = criterion(d_fake, torch.ones_like(d_fake))

            opt_g.zero_grad()
            g_loss.backward()
            opt_g.step()

            epoch_d_loss += d_loss.item()
            epoch_g_loss += g_loss.item()
            n_batches += 1

        avg_d = epoch_d_loss / n_batches
        avg_g = epoch_g_loss / n_batches
        history["d_loss"].append(avg_d)
        history["g_loss"].append(avg_g)

        # Save sample images every 10 epochs
        if epoch % 10 == 0 or epoch == 1:
            with torch.no_grad():
                z = torch.randn(64, LATENT_DIM).to(device)
                samples = G(z).cpu().view(-1, 28, 28).numpy()
            history["samples"].append((epoch, samples))
            print(f"Epoch {epoch}/{n_epochs} | D loss: {avg_d:.4f} | G loss: {avg_g:.4f}")

    return G, D, history


G_mnist, D_mnist, history_mnist = train_mnist_gan(n_epochs=50, k=1)

### Figure 2 재현: 생성 샘플 시각화 / Reproducing Figure 2: Generated Samples

논문의 Figure 2처럼 학습된 Generator로부터 생성된 MNIST 숫자를 시각화합니다.
샘플은 cherry-pick되지 않은 fair random draws입니다.

Like Figure 2, we visualize MNIST digits generated from the trained Generator.
Samples are fair random draws, not cherry-picked.

In [ ]:
# --- Visualize generated MNIST samples at different epochs ---

fig, axes = plt.subplots(len(history_mnist["samples"]), 8, figsize=(12, 2 * len(history_mnist["samples"])))
if len(history_mnist["samples"]) == 1:
    axes = axes[np.newaxis, :]

for row, (epoch, samples) in enumerate(history_mnist["samples"]):
    for col in range(8):
        axes[row, col].imshow(samples[col], cmap='gray')
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'Epoch {epoch}', fontsize=10, rotation=0, labelpad=50)

fig.suptitle("Generated MNIST Samples During Training", fontsize=13)
plt.tight_layout()
plt.show()

### Figure 3 재현: 잠재 공간 보간 / Reproducing Figure 3: Latent Space Interpolation

논문의 Figure 3처럼 잠재 공간 $z$에서 두 점 사이를 선형 보간하여
생성 이미지가 부드럽게 전환되는지 확인합니다.
이는 Generator가 의미 있는 연속적 표현을 학습했음을 시사합니다.

Like Figure 3, we linearly interpolate between two points in latent space $z$
to verify smooth transitions, suggesting the Generator learned meaningful
continuous representations.

In [ ]:
# --- Latent space interpolation (Figure 3) ---

def interpolate_latent(G, z1, z2, n_steps: int = 10):
    """Linearly interpolate between two latent vectors and generate images."""
    alphas = np.linspace(0, 1, n_steps)
    images = []
    with torch.no_grad():
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            img = G(z).cpu().view(28, 28).numpy()
            images.append(img)
    return images


# Generate 3 rows of interpolation
fig, axes = plt.subplots(3, 10, figsize=(14, 4.5))

for row in range(3):
    z1 = torch.randn(1, LATENT_DIM).to(device)
    z2 = torch.randn(1, LATENT_DIM).to(device)
    images = interpolate_latent(G_mnist, z1, z2, n_steps=10)

    for col, img in enumerate(images):
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')

fig.suptitle("Latent Space Interpolation (Figure 3 reproduction)", fontsize=13)
plt.tight_layout()
plt.show()

## Part 3: 이론 검증 / Theory Verification

논문의 핵심 이론적 결과를 수치적으로 검증합니다:
We numerically verify the paper's key theoretical results:

1. **Proposition 1**: 최적 Discriminator $D^*(x) = \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}$
2. **Theorem 1**: 전역 최솟값 $C(G) = -\log 4$이며 $p_g = p_{\text{data}}$일 때 달성
3. **Eq. 6**: $C(G) = -\log(4) + 2 \cdot JSD(p_{\text{data}} \| p_g)$

In [ ]:
# --- Theory verification: Optimal D and JSD ---
from scipy.stats import norm

def compute_optimal_d(x, p_data_vals, p_g_vals):
    """Compute optimal discriminator D*(x) = p_data / (p_data + p_g) (Eq. 2)."""
    return p_data_vals / (p_data_vals + p_g_vals + 1e-10)


def compute_jsd(p, q, x, dx):
    """Compute Jensen-Shannon Divergence between distributions p and q.

    JSD(p||q) = 0.5 * KL(p||m) + 0.5 * KL(q||m), where m = (p+q)/2
    """
    m = 0.5 * (p + q)
    # Avoid log(0) by masking zero values
    mask_p = (p > 1e-10) & (m > 1e-10)
    mask_q = (q > 1e-10) & (m > 1e-10)

    kl_pm = np.sum(p[mask_p] * np.log(p[mask_p] / m[mask_p])) * dx
    kl_qm = np.sum(q[mask_q] * np.log(q[mask_q] / m[mask_q])) * dx
    return 0.5 * kl_pm + 0.5 * kl_qm


# Set up distributions
x_range = np.linspace(-5, 15, 10000)
dx = x_range[1] - x_range[0]
p_data_vals = norm.pdf(x_range, loc=DATA_MU, scale=DATA_SIGMA)

# Simulate p_g at different training stages
# p_g starts far from p_data and gradually converges
p_g_stages = {
    "Initial (far)": norm.pdf(x_range, loc=0, scale=2.0),
    "Mid-training": norm.pdf(x_range, loc=3, scale=1.5),
    "Near convergence": norm.pdf(x_range, loc=3.8, scale=1.3),
    "Converged": norm.pdf(x_range, loc=DATA_MU, scale=DATA_SIGMA),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (stage_name, p_g_vals) in enumerate(p_g_stages.items()):
    ax = axes[idx]

    # Compute optimal D* (Proposition 1)
    d_star = compute_optimal_d(x_range, p_data_vals, p_g_vals)

    # Compute JSD (Eq. 6)
    jsd = compute_jsd(p_data_vals, p_g_vals, x_range, dx)
    c_g = -np.log(4) + 2 * jsd

    # Plot
    ax.plot(x_range, p_data_vals, 'k:', linewidth=2, label=r'$p_{\mathrm{data}}$')
    ax.plot(x_range, p_g_vals, 'g-', linewidth=2, alpha=0.7, label=r'$p_g$')

    ax2 = ax.twinx()
    ax2.plot(x_range, d_star, 'b--', linewidth=1.5, label=r'$D^*(x)$')
    ax2.set_ylim(-0.1, 1.1)
    ax2.axhline(y=0.5, color='gray', linestyle=':', alpha=0.3)
    ax2.set_ylabel('$D^*(x)$', color='blue')

    ax.set_title(f"{stage_name}\nJSD = {jsd:.4f} | C(G) = {c_g:.4f}")
    ax.set_xlim(-5, 13)
    ax.legend(loc='upper left', fontsize=9)
    ax.set_xlabel('$x$')
    ax.set_ylabel('Density')

fig.suptitle("Theory Verification: Optimal D* (Eq. 2) and JSD (Eq. 6)\n"
             r"Theorem 1: $C^* = -\log 4 \approx$ " + f"{-np.log(4):.4f}",
             fontsize=13)
plt.tight_layout()
plt.show()

print(f"Theoretical minimum: C* = -log(4) = {-np.log(4):.4f}")
print(f"At convergence (p_g = p_data): JSD = {compute_jsd(p_data_vals, p_data_vals, x_range, dx):.6f}")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| **Generator** | MLP (ReLU + sigmoid) | CNN-based (DCGAN, StyleGAN) or Transformer-based |
| **Discriminator** | MLP (maxout + dropout) | CNN with spectral normalization |
| **Optimizer** | SGD with momentum | Adam with $\beta_1 = 0.5$ |
| **Loss function** | BCE (minimax game, Eq. 1) | Wasserstein loss (WGAN), hinge loss |
| **Training trick** | $\max \log D(G(z))$ instead of $\min \log(1-D(G(z)))$ | Non-saturating loss (standard practice) |
| **Evaluation** | Parzen window log-likelihood | FID (Frechet Inception Distance), IS (Inception Score) |
| **Architecture** | Fully connected MLP | Deep CNNs, Progressive growing, Style-based |
| **Noise prior** | Uniform or Gaussian | Gaussian (standard practice) |

### 핵심 구현 교훈 / Key Implementation Lessons

1. **D와 G의 균형이 핵심** — D가 너무 강하면 G의 gradient가 사라지고, G가 너무 강하면 mode collapse 발생 / Balance between D and G is critical
2. **$k=1$이면 충분** — 논문에서도 k=1을 사용했고, 실제로도 잘 작동함 / k=1 is sufficient as shown in the paper
3. **Gradient saturation 트릭은 필수** — 이론적 목적함수와 실전 목적함수가 다름을 인식해야 함 / The gradient saturation trick is essential — theoretical and practical objectives differ
4. **생성 품질 평가가 어렵다** — 2014년에는 Parzen window뿐이었으나, 현재는 FID가 표준 / Evaluating generation quality is hard — only Parzen window in 2014, now FID is standard